In [26]:
# 智谱ai
key = '1548e902fbc7499eae569cac6237f8c1.xjmOzH0nfaaSmjxb'

import os
from langchain_community.chat_models import ChatZhipuAI

os.environ["ZHIPUAI_API_KEY"] = key

chat = ChatZhipuAI(
    model="glm-4.6v",
    streaming=True,
    max_tokens=512,
    temperature=0.5,
)


In [27]:
#提示词模板
from langchain.prompts import PromptTemplate, prompt

#定义模板
template = '你是一个{role}，请用{style}风格回答问题：{question}'
#创建模板对象
prompt_template = PromptTemplate.from_template(template)

#变量填充
filled_prompt = prompt_template.format(role='数学教师',style='详细',question='勾股定理是什么？')

#响应数据
AI_msg = chat.invoke(filled_prompt)
print(AI_msg.content)

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import display, HTML

#定义模板
sys_template = '你是一个老师，请以{style}风格回答问题'
user_template = '请用简洁的方式介绍：{question}'

#创建对话模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system", sys_template),
    ("human", user_template),
])

#填充变量
prompt = prompt_template.format_messages(style='简明易懂',question='介绍你自己')

#获取响应
response = chat.invoke(prompt)
display(HTML(f"<div style='white-space: pre-wrap; word-break: break-word;'>{response}</div>"))

In [35]:
#输出格式化
from langchain import output_parsers
from langchain.output_parsers import StructuredOutputParser,ResponseSchema



# 定义输出结构
response_schemas = [
    ResponseSchema(name="name",description="人的姓名",type='string'),
    ResponseSchema(name="age",description="人的年龄",typr='integer'),
]
#输出解析器
output_parsers = StructuredOutputParser.from_response_schemas(response_schemas)

template = """你是一个信息提取助手，请从以下文本提取姓名和年龄，并以JSON形式返回：
文本：{input_text}
{format_insructions}"""
#创建模板对象
prompt_template = PromptTemplate.from_template(
    template=template,
    partial_variables={"format_insructions":output_parsers.get_format_instructions()}
)

#变量填充
input_text = "张三今年25岁，来自北京。"
filled_prompt = prompt_template.format(input_text=input_text)

#响应数据
res = chat.invoke(filled_prompt)
parsed_output = output_parsers.parse(res.content)
print(parsed_output)


{'name': '张三', 'age': '25'}


In [36]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

#定义输出模板
class Person(BaseModel):
    name:str = Field(description="人的姓名")
    age:int = Field(description="人的年龄")

#创建输出解释器
output_parsers = PydanticOutputParser(pydantic_object=Person)


template = """你是一个信息提取助手，请从以下文本提取姓名和年龄，并以JSON形式返回：
文本：{input_text}
{format_insructions}"""
#创建模板对象
prompt_template = PromptTemplate.from_template(
    template=template,
    partial_variables={"format_insructions":output_parsers.get_format_instructions()}
)

#变量填充
input_text = "李四今年30岁，来自上海。"
filled_prompt = prompt_template.format(input_text=input_text)

#获取输出
resp = chat.invoke(filled_prompt)
parsed_output = output_parsers.parse(resp.content)
#print(parsed_output)
parsed_output

Person(name='李四', age=30)

In [44]:
from langchain.output_parsers import StructuredOutputParser,ResponseSchema
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

#链式调用
response_schemas = [
    ResponseSchema(name="answer",description="问题的答案",type='string'),
    ResponseSchema(name="confidence",description="答案的置信度",typr='float'),
]
#输出解析器
output_parsers = StructuredOutputParser.from_response_schemas(response_schemas)

template = """你是一个数学老师，请用{style}风格回答以下问题，并以JSON形式返回答案和置信度：
文本：{question}
{format_insructions}"""
#创建模板对象
prompt_template = PromptTemplate.from_template(
    template=template,
    partial_variables={"format_insructions":output_parsers.get_format_instructions()}
)
#创建LLMChain
#llm_chain = LLMChain(llm=chat,prompt=prompt_template,output_parser=output_parsers)
llm_chain = prompt_template | chat | output_parsers #推荐的写法

#执行链
res = llm_chain.invoke({
    'style':'通俗易懂',
    'question':'勾股定理是什么？'
})
print(res)

{'answer': '勾股定理说的是，在直角三角形里，两条直角边（也就是组成直角的两条边）的长度分别平方后加起来的结果，等于斜边（直角所对的边，也就是最长的边）长度的平方。简单说，就是‘直角边平方和 = 斜边平方’。比如，一个直角三角形的两条直角边分别是3和4，那么3的平方（9）加上4的平方（16）等于25，而斜边的长度就是5（因为5的平方是25），这样就能验证这个定理啦～', 'confidence': '极高（勾股定理是数学中非常基础且被广泛验证的定理，解释符合其定义）'}


In [47]:
#流式输出
from langchain.output_parsers import StructuredOutputParser,ResponseSchema
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

response_schemas = [
    ResponseSchema(name="answer",description="问题的答案",type='string'),
    ResponseSchema(name="confidence",description="答案的置信度",typr='float'),
]
#输出解析器
output_parsers = StructuredOutputParser.from_response_schemas(response_schemas)

template = """你是一个数学老师，请用{style}风格回答以下问题，并以JSON形式返回答案和置信度：
文本：{question}
{format_insructions}"""
#创建模板对象
prompt_template = PromptTemplate.from_template(
    template=template,
    partial_variables={"format_insructions":output_parsers.get_format_instructions()}
)
#创建LLMChain
#llm_chain = LLMChain(llm=chat,prompt=prompt_template,output_parser=output_parsers)
llm_chain = prompt_template | chat #推荐的写法

#执行链
chunks = [] #存储输出
for chunk in llm_chain.stream({
    'style':'通俗易懂',
    'question':'勾股定理是什么？'
}):
    chunks.append(chunk) #存储每次输出
    print(chunk.content,end='',flush=True)


```json
{
	"answer": "勾股定理说的是，在直角三角形里，两条直角边（组成直角的两条边）的长度各自平方后加起来，等于斜边（最长的那条边）长度的平方。简单理解就是：直角三角形的两条“短边”平方和，等于“长边”的平方，比如直角边是3和4，斜边就是5，因为3²+4²=9+16=25=5²~",
	"confidence": "高（基于数学定理的确定性）"
}
```